<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Pipeline_(Heatmap).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import tensorflow as tf
from google.colab import drive

drive.mount('/content/drive')

# Load your trained model
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Wafer-detect.keras"
)

# Wake the model (VERY IMPORTANT)
_ = model.predict(tf.zeros([1,224,224,3]))
print("Model loaded and ready ✔️")

# Your class names (must match training order)
class_names = ['bridge','clean','cmp','crack','ler','open','others','vias']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ValueError: File not found: filepath=/content/drive/MyDrive/Wafer-detect.keras. Please ensure the file is an accessible `.keras` zip file.

CELL 2 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


CELL 3 — Load grayscale image safely (convert to RGB)

In [ ]:
def load_image(path):
    img = tf.keras.utils.load_img(
        path, target_size=(224,224), color_mode="grayscale"
    )
    arr = tf.keras.utils.img_to_array(img)      # (224,224,1)
    arr = np.repeat(arr, 3, axis=-1)           # → (224,224,3)
    arr = arr / 255.0
    arr = np.expand_dims(arr, axis=0)
    return arr, img


CELL 4 — Pick last conv layer automatically

In [ ]:
base_model = model.layers[0]  # MobileNet backbone

last_conv_layer = [l.name for l in base_model.layers
                   if "conv" in l.name][-1]

print("Using layer:", last_conv_layer)


CELL 5 — SIMPLE CAM (reliable heatmap, no gradient errors)

In [ ]:
def simple_cam(model, img_array):
    base = model.layers[0]

    feature_model = tf.keras.Model(
        inputs=base.input,
        outputs=base.get_layer(last_conv_layer).output
    )

    features = feature_model(img_array)[0]   # (7,7,channels)

    # average channels → importance map
    heatmap = tf.reduce_mean(features, axis=-1)

    # normalize 0–1
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap)

    return heatmap.numpy()



CELL 6 — Robust overlay (won’t crash)

In [ ]:
def overlay_heatmap(heatmap, img):

    heatmap = np.array(heatmap, dtype=np.float32)

    if heatmap.ndim == 1:
        heatmap = heatmap.reshape((7,7))

    heatmap = cv2.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img = np.array(img)
    return cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)


CELL 7 — ONE‑CLICK DEMO (change image path only)

In [ ]:
# -------- CHANGE THIS TO YOUR IMAGE ----------
IMG_PATH = "/content/drive/MyDrive/Datasets/test/bridge/samplebridge.png"
# --------------------------------------------

img_array, original_img = load_image(IMG_PATH)

# Prediction
pred = model.predict(img_array)
pred_class = np.argmax(pred)
print("Predicted defect:", class_names[pred_class])

# Heatmap
heatmap = simple_cam(model, img_array)
result = overlay_heatmap(heatmap, original_img)

# Show results
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(original_img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(result)
plt.title("Heatmap")
plt.axis("off")

plt.show()
